In [11]:
import pandas as pd
from pandas import DataFrame, read_csv
import kagglehub
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import shutil


In [12]:
df = read_csv("data/raw/archive/full_df.csv")
df.head()

,ID,Patient Age,Patient Sex,Left-Fundus,Right-Fundus,Left-Diagnostic Keywords,Right-Diagnostic Keywords,N,D,G,C,A,H,M,O,filepath,labels,target,filename
0,0,69,Female,0_left.jpg,0_right.jpg,cataract,normal fundus,0,0,0,1,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...,['N'],"[1, 0, 0, 0, 0, 0, 0, 0]",0_right.jpg
1,1,57,Male,1_left.jpg,1_right.jpg,normal fundus,normal fundus,1,0,0,0,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...,['N'],"[1, 0, 0, 0, 0, 0, 0, 0]",1_right.jpg
2,2,42,Male,2_left.jpg,2_right.jpg,laser spot，moderate non proliferative retinopathy,moderate non proliferative retinopathy,0,1,0,0,0,0,0,1,../input/ocular-disease-recognition-odir5k/ODI...,['D'],"[0, 1, 0, 0, 0, 0, 0, 0]",2_right.jpg
3,4,53,Male,4_left.jpg,4_right.jpg,macular epiretinal membrane,mild nonproliferative retinopathy,0,1,0,0,0,0,0,1,../input/ocular-disease-recognition-odir5k/ODI...,['D'],"[0, 1, 0, 0, 0, 0, 0, 0]",4_right.jpg
4,5,50,Female,5_left.jpg,5_right.jpg,moderate non proliferative retinopathy,moderate non proliferative retinopathy,0,1,0,0,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...,['D'],"[0, 1, 0, 0, 0, 0, 0, 0]",5_right.jpg


In [13]:
df.columns

Index(['ID', 'Patient Age', 'Patient Sex', 'Left-Fundus', 'Right-Fundus',
       'Left-Diagnostic Keywords', 'Right-Diagnostic Keywords', 'N', 'D', 'G',
       'C', 'A', 'H', 'M', 'O', 'filepath', 'labels', 'target', 'filename'],
      dtype='object')

In [14]:
df = df[df["labels"].str.contains("N|M")]


In [15]:
df["labels"].unique()

array(["['N']", "['M']"], dtype=object)

In [16]:
df = df[(df["N"] == 1) | (df["M"] == 1)]

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2359 entries, 1 to 6356
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   ID                         2359 non-null   int64 
 1   Patient Age                2359 non-null   int64 
 2   Patient Sex                2359 non-null   object
 3   Left-Fundus                2359 non-null   object
 4   Right-Fundus               2359 non-null   object
 5   Left-Diagnostic Keywords   2359 non-null   object
 6   Right-Diagnostic Keywords  2359 non-null   object
 7   N                          2359 non-null   int64 
 8   D                          2359 non-null   int64 
 9   G                          2359 non-null   int64 
 10  C                          2359 non-null   int64 
 11  A                          2359 non-null   int64 
 12  H                          2359 non-null   int64 
 13  M                          2359 non-null   int64 
 14  O            

In [17]:
columns_to_keep = [ "ID","Left-Fundus", "Right-Fundus", "N", "M", "filepath", "labels", "target", "filename"]
df = df[columns_to_keep]

df.head()

,ID,Left-Fundus,Right-Fundus,N,M,filepath,labels,target,filename
1,1,1_left.jpg,1_right.jpg,1,0,../input/ocular-disease-recognition-odir5k/ODI...,['N'],"[1, 0, 0, 0, 0, 0, 0, 0]",1_right.jpg
7,8,8_left.jpg,8_right.jpg,1,0,../input/ocular-disease-recognition-odir5k/ODI...,['N'],"[1, 0, 0, 0, 0, 0, 0, 0]",8_right.jpg
11,13,13_left.jpg,13_right.jpg,0,1,../input/ocular-disease-recognition-odir5k/ODI...,['M'],"[0, 0, 0, 0, 0, 0, 1, 0]",13_right.jpg
14,16,16_left.jpg,16_right.jpg,0,1,../input/ocular-disease-recognition-odir5k/ODI...,['M'],"[0, 0, 0, 0, 0, 0, 1, 0]",16_right.jpg
16,18,18_left.jpg,18_right.jpg,0,1,../input/ocular-disease-recognition-odir5k/ODI...,['M'],"[0, 0, 0, 0, 0, 0, 1, 0]",18_right.jpg


In [18]:
df["N"].value_counts()

N
1    2101
0     258
Name: count, dtype: int64

Imbalance will use class weight


In [19]:
import os
from pathlib import Path
from PIL import Image, ImageOps, ImageFilter
import numpy as np
from torchvision import transforms

image_dir = Path("data/raw/archive/ODIR-5K/ODIR-5K/Training Images")  
output_dir = Path("data/cleaned/preprocessed_images")  
output_dir.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = (512, 512)  
NORMALIZE = True  

augmentation = transforms.Compose([
    transforms.RandomRotation(15),          
    transforms.RandomHorizontalFlip(),      
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  
])

def resize_with_padding(image, size=(512, 512)):
    """
    Resize an image while preserving the aspect ratio by adding padding.
    
    Args:
        image (PIL.Image): Input image.
        size (tuple): Target size (width, height).
    
    Returns:
        PIL.Image: Resized image with padding.
    """
    image.thumbnail(size, Image.Resampling.LANCZOS)
    
    new_image = Image.new("RGB", size, (255, 255, 255))
    
    new_image.paste(image, ((size[0] - image.size[0]) // 2, (size[1] - image.size[1]) // 2))
    
    return new_image

def sharpen_image(image):
    """
    Apply sharpening to an image.
    
    Args:
        image (PIL.Image): Input image.
    
    Returns:
        PIL.Image: Sharpened image.
    """
    return image.filter(ImageFilter.SHARPEN)

def upscale_image(image, scale=2):
    """
    Upscale an image using OpenCV.
    
    Args:
        image (PIL.Image): Input image.
        scale (int): Upscaling factor.
    
    Returns:
        PIL.Image: Upscaled image.
    """
    import cv2
    image_np = np.array(image)
    
    upscaled = cv2.resize(image_np, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    
    return Image.fromarray(upscaled)

def preprocess_image(image_path, output_path, augment=False):
    """
    Preprocess a single image: resize, normalize, sharpen, and optionally augment.
    
    Args:
        image_path (Path): Path to the input image.
        output_path (Path): Path to save the preprocessed image.
        augment (bool): Whether to apply data augmentation.
    """
    try:
        image = Image.open(image_path)
        
        image = upscale_image(image, scale=2)
        
        image = resize_with_padding(image, IMAGE_SIZE)
        
        image = sharpen_image(image)
        
        if augment:
            image = augmentation(image)
        
        if NORMALIZE:
            image = np.array(image) / 255.0
            image = Image.fromarray((image * 255).astype(np.uint8)) 
        
        image.save(output_path)
    except Exception as e:
        print(f"Error processing image {image_path}: {e}")

def preprocess_images_from_df(df, augment=False):
    """
    Preprocess images listed in the cleaned DataFrame.
    
    Args:
        df (pd.DataFrame): DataFrame containing image paths and labels.
        augment (bool): Whether to apply data augmentation.
    """
    for _, row in df.iterrows():
        for eye in ["Left-Fundus", "Right-Fundus"]:
            input_path = image_dir / row[eye]
            output_path = output_dir / row[eye]
            
            output_path.parent.mkdir(parents=True, exist_ok=True)
            
            # Preprocess the image
            preprocess_image(input_path, output_path, augment=augment)

if __name__ == "__main__":
    import pandas as pd


    print("Preprocessing images from the cleaned DataFrame...")
    preprocess_images_from_df(df, augment=True)

    print("Preprocessing complete!")

Preprocessing images from the cleaned DataFrame...
Preprocessing complete!


In [20]:
df["N"].value_counts()

N
1    2101
0     258
Name: count, dtype: int64

In [21]:
from pathlib import Path

output_folder = Path("data/cleaned")
output_folder.mkdir(parents=True, exist_ok=True) 
output_file = output_folder / "cleaned_odir_N_M.csv"

df.to_csv(output_file, index=False)

print(f"DataFrame saved to {output_file}")

DataFrame saved to data/cleaned/cleaned_odir_N_M.csv
